<a href="https://colab.research.google.com/github/kowsik005/Unsupervised_ML_project/blob/main/Association_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ============================================================
# ASSOCIATION RULE MINING - APRIORI
# UNSUPERVISED LEARNING
# GOOGLE COLAB
# ============================================================

# ------------------------------------------------------------
# 1. HIDE WARNINGS
# ------------------------------------------------------------

import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


# ------------------------------------------------------------
# 2. IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules


print("Libraries imported successfully!")


# ------------------------------------------------------------
# 3. LOAD DATASET
# ------------------------------------------------------------

file_path = "/content/Market_Basket_Optimisation-selected-columns.csv"

df = pd.read_csv(file_path, header=None)

print("\nDataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
display(df.head())


# ------------------------------------------------------------
# 4. CREATE TRANSACTIONS
# ------------------------------------------------------------

transactions = []

for _, row in df.iterrows():

    transaction = []

    for item in row:

        if pd.notna(item):

            item = str(item).strip()

            if item != "":
                transaction.append(item)

    if len(transaction) > 0:
        transactions.append(transaction)


print("\nNumber of Transactions:")
print(len(transactions))

print("\nExample Transaction:")
print(transactions[0])


# ------------------------------------------------------------
# 5. ONE-HOT ENCODING
# ------------------------------------------------------------

te = TransactionEncoder()

encoded_data = te.fit(
    transactions
).transform(
    transactions
)

basket = pd.DataFrame(
    encoded_data,
    columns=te.columns_
)

print("\nOne-Hot Encoded Dataset:")
display(basket.head())

print("\nBasket Shape:")
print(basket.shape)


# ------------------------------------------------------------
# 6. APRIORI ALGORITHM
# ------------------------------------------------------------

min_support = 0.05

frequent_itemsets = apriori(
    basket,
    min_support=min_support,
    use_colnames=True
)

print(
    "\nNumber of Frequent Itemsets:",
    len(frequent_itemsets)
)


# ------------------------------------------------------------
# 7. DISPLAY FREQUENT ITEMSETS
# ------------------------------------------------------------

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

print(
    "\n========== FREQUENT ITEMSETS =========="
)

display(
    frequent_itemsets.head(20)
)


# ------------------------------------------------------------
# 8. GENERATE ASSOCIATION RULES
# ------------------------------------------------------------

if len(frequent_itemsets) > 0:

    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0.30
    )

else:

    rules = pd.DataFrame()


# ------------------------------------------------------------
# 9. CHECK RULES
# ------------------------------------------------------------

if len(rules) == 0:

    print("\nNo association rules found.")

    print(
        "Try changing min_support from 0.05 to 0.02."
    )

else:

    print(
        "\nNumber of Association Rules:",
        len(rules)
    )


# ------------------------------------------------------------
# 10. SELECT ONLY REQUIRED COLUMNS
# ------------------------------------------------------------

if len(rules) > 0:

    rules = rules[
        [
            "antecedents",
            "consequents",
            "support",
            "confidence",
            "lift"
        ]
    ].copy()


# ------------------------------------------------------------
# 11. CONVERT FROZENSET TO TEXT
# ------------------------------------------------------------

if len(rules) > 0:

    rules["antecedents"] = rules[
        "antecedents"
    ].apply(
        lambda x: ", ".join(sorted(x))
    )

    rules["consequents"] = rules[
        "consequents"
    ].apply(
        lambda x: ", ".join(sorted(x))
    )


# ------------------------------------------------------------
# 12. REMOVE INVALID VALUES
# ------------------------------------------------------------

if len(rules) > 0:

    rules = rules.replace(
        [np.inf, -np.inf],
        np.nan
    )

    rules = rules.dropna(
        subset=[
            "support",
            "confidence",
            "lift"
        ]
    )


# ------------------------------------------------------------
# 13. ROUND VALUES
# ------------------------------------------------------------

if len(rules) > 0:

    rules["support"] = rules[
        "support"
    ].round(4)

    rules["confidence"] = rules[
        "confidence"
    ].round(4)

    rules["lift"] = rules[
        "lift"
    ].round(4)


# ------------------------------------------------------------
# 14. SORT BY LIFT
# ------------------------------------------------------------

if len(rules) > 0:

    rules = rules.sort_values(
        by="lift",
        ascending=False
    )


# ------------------------------------------------------------
# 15. DISPLAY TOP RULES
# ------------------------------------------------------------

if len(rules) > 0:

    print(
        "\n========== TOP ASSOCIATION RULES =========="
    )

    display(
        rules.head(20)
    )


# ------------------------------------------------------------
# 16. STRONG RULES
# ------------------------------------------------------------

if len(rules) > 0:

    strong_rules = rules[
        (rules["confidence"] >= 0.50)
        &
        (rules["lift"] > 1)
    ]

    print(
        "\n========== STRONG ASSOCIATION RULES =========="
    )

    print(
        "Number of Strong Rules:",
        len(strong_rules)
    )

    display(
        strong_rules.head(20)
    )


# ------------------------------------------------------------
# 17. BEST RULE
# ------------------------------------------------------------

if len(rules) > 0:

    best_rule = rules.iloc[0]

    print(
        "\n========== BEST ASSOCIATION RULE =========="
    )

    print(
        "\nIF CUSTOMER BUYS:"
    )

    print(
        best_rule["antecedents"]
    )

    print(
        "\nTHEN CUSTOMER MAY ALSO BUY:"
    )

    print(
        best_rule["consequents"]
    )

    print(
        "\nSupport:",
        best_rule["support"]
    )

    print(
        "Confidence:",
        best_rule["confidence"]
    )

    print(
        "Lift:",
        best_rule["lift"]
    )


# ------------------------------------------------------------
# 18. TOP 10 BY CONFIDENCE
# ------------------------------------------------------------

if len(rules) > 0:

    print(
        "\n========== TOP 10 BY CONFIDENCE =========="
    )

    top_confidence = (
        rules
        .sort_values(
            by="confidence",
            ascending=False
        )
        .head(10)
    )

    display(
        top_confidence
    )


# ------------------------------------------------------------
# 19. TOP 10 BY LIFT
# ------------------------------------------------------------

if len(rules) > 0:

    print(
        "\n========== TOP 10 BY LIFT =========="
    )

    top_lift = (
        rules
        .sort_values(
            by="lift",
            ascending=False
        )
        .head(10)
    )

    display(
        top_lift
    )


# ------------------------------------------------------------
# 20. SAVE RESULTS
# ------------------------------------------------------------

frequent_itemsets.to_csv(
    "/content/frequent_itemsets.csv",
    index=False
)

if len(rules) > 0:

    rules.to_csv(
        "/content/association_rules.csv",
        index=False
    )

print(
    "\n========== FILES SAVED =========="
)

print(
    "Frequent Itemsets:"
)

print(
    "/content/frequent_itemsets.csv"
)

if len(rules) > 0:

    print(
        "\nAssociation Rules:"
    )

    print(
        "/content/association_rules.csv"
    )


# ------------------------------------------------------------
# 21. FINAL INTERPRETATION
# ------------------------------------------------------------

print("""
============================================================
ASSOCIATION RULE MINING COMPLETED
============================================================

Algorithm:
Apriori

Learning Type:
Unsupervised Learning

Important Measures:

1. SUPPORT
   How frequently items occur together.

2. CONFIDENCE
   Probability of buying the consequent
   when the antecedent is purchased.

3. LIFT
   Strength of the relationship between items.

Interpretation:

Lift > 1  = Positive association
Lift = 1  = No significant association
Lift < 1  = Negative association

Example:

Bread -> Milk

Support = 0.10
Confidence = 0.60
Lift = 2.00

Meaning:

10% of transactions contain both products.

60% of customers who bought Bread
also bought Milk.

Customers who bought Bread are
approximately 2 times more likely
to buy Milk.

============================================================
""")


Libraries imported successfully!

Dataset Shape:
(125, 10)

First 5 Rows:


,0,1,2,3,4,5,6,7,8,9
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Number of Transactions:
1

Example Transaction:
['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice']

One-Hot Encoded Dataset:


,almonds,avocado,cottage cheese,energy drink,green grapes,shrimp,tomato juice,vegetables mix,whole weat flour,yams
0,True,True,True,True,True,True,True,True,True,True



Basket Shape:
(1, 10)

Number of Frequent Itemsets: 1023

========== FREQUENT ITEMSETS ==========


...


========== BEST ASSOCIATION RULE ==========

IF CUSTOMER BUYS:
tomato juice

THEN CUSTOMER MAY ALSO BUY:
almonds

Support: 1.0
Confidence: 1.0
Lift: 1.0

========== TOP 10 BY CONFIDENCE ==========


,antecedents,consequents,support,confidence,lift
10,"almonds, cottage cheese, energy drink, green g...","avocado, whole weat flour",1.0,1.0,1.0
57001,tomato juice,almonds,1.0,1.0,1.0
0,"almonds, avocado, cottage cheese, energy drink...",whole weat flour,1.0,1.0,1.0
1,"almonds, cottage cheese, energy drink, green g...",avocado,1.0,1.0,1.0
2,"almonds, avocado, cottage cheese, energy drink...",green grapes,1.0,1.0,1.0
3,"almonds, avocado, energy drink, green grapes, ...",cottage cheese,1.0,1.0,1.0
4,"almonds, avocado, cottage cheese, green grapes...",energy drink,1.0,1.0,1.0
5,"almonds, avocado, cottage cheese, energy drink...",tomato juice,1.0,1.0,1.0
6,"avocado, cottage cheese, energy drink, green g...",almonds,1.0,1.0,1.0
7,"almonds, avocado, cottage cheese, energy drink...",yams,1.0,1.0,1.0



========== TOP 10 BY LIFT ==========


,antecedents,consequents,support,confidence,lift
10,"almonds, cottage cheese, energy drink, green g...","avocado, whole weat flour",1.0,1.0,1.0
57001,tomato juice,almonds,1.0,1.0,1.0
0,"almonds, avocado, cottage cheese, energy drink...",whole weat flour,1.0,1.0,1.0
1,"almonds, cottage cheese, energy drink, green g...",avocado,1.0,1.0,1.0
2,"almonds, avocado, cottage cheese, energy drink...",green grapes,1.0,1.0,1.0
3,"almonds, avocado, energy drink, green grapes, ...",cottage cheese,1.0,1.0,1.0
4,"almonds, avocado, cottage cheese, green grapes...",energy drink,1.0,1.0,1.0
5,"almonds, avocado, cottage cheese, energy drink...",tomato juice,1.0,1.0,1.0
6,"avocado, cottage cheese, energy drink, green g...",almonds,1.0,1.0,1.0
7,"almonds, avocado, cottage cheese, energy drink...",yams,1.0,1.0,1.0



========== FILES SAVED ==========
Frequent Itemsets:
/content/frequent_itemsets.csv

Association Rules:
/content/association_rules.csv

ASSOCIATION RULE MINING COMPLETED

Algorithm:
Apriori

Learning Type:
Unsupervised Learning

Important Measures:

1. SUPPORT
   How frequently items occur together.

2. CONFIDENCE
   Probability of buying the consequent
   when the antecedent is purchased.

3. LIFT
   Strength of the relationship between items.

Interpretation:

Lift > 1  = Positive association
Lift = 1  = No significant association
Lift < 1  = Negative association

Example:

Bread -> Milk

Support = 0.10
Confidence = 0.60
Lift = 2.00

Meaning:

10% of transactions contain both products.

60% of customers who bought Bread
also bought Milk.

Customers who bought Bread are
approximately 2 times more likely
to buy Milk.


